In [1]:
%matplotlib inline
from pathlib import Path
import click
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.backends.backend_pdf
import matplotlib.style as mplstyle
import os
from datetime import datetime
import matplotlib.colors as mcolors

from matplotlib.collections import LineCollection, PolyCollection

In [2]:
from hole_trajectory import calculate_trajectory_3d, plot_trajectory_3d, calc_binned_angles, plot_stability_comparison, calc_inclination, plot_inclination

In [3]:
mplstyle.use('fast')
plt.rcParams['lines.markersize'] = 1
plt.rcParams['font.size'] = 14
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['image.cmap'] = "viridis"
plt.rcParams['figure.max_open_warning'] = 50

In [4]:
# Publication-quality settings for this figure
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

In [5]:
from multi_log_reader import MultiLogReader
from preprocess import preprocess
#folder = "/Users/delia/Library/CloudStorage/OneDrive-SharedLibraries-NERC/BAS BigRAID Documents/Season Reports/2025/DataLog" 


In [6]:
def _plot_stability(df_comp, subtitle):
    comp_mean_x, comp_std_x, _ = calc_binned_angles(df_comp, "hole_pitch", dz=dz)
    comp_mean_y, comp_std_y, _ = calc_binned_angles(df_comp, "hole_roll", dz=dz)

    comp_z_coords, comp_x_mean, comp_x_std, comp_y_mean, comp_y_std = calculate_trajectory_3d(
        comp_mean_x, comp_std_x,
        comp_mean_y, comp_std_y
    )


In [7]:
def plot_trajectory_coordinates(df, out_path):
    df = preprocess(df, run_depth_threshold=1.5)
    # extract site specific label
    string_path = "%s" % out_path
    string_path = string_path.split('/')[1].replace('_inclination.pdf', "")
    print (string_path)
    figures_start_idx = plt.gcf().number + 1
    display_max = 0
    for i, k in enumerate(["[PLC]IMUYAW","[PLC]IMUPITCH", "[PLC]IMUROLL"]):
        fig = plt.figure(figsize=(5, 5))
        ax = plt.subplot()
        d = df[(df["[PLC]WIRESPOOLEDOUT"] > 2)]
        # subtract the mean when the drill is hanging freely above the hole
        #zero_offset = df[(df["[PLC]WIRESPOOLEDOUT"] < 1) & (df["[PLC]CABLESPEED"].abs() < 0.1) & (df["[PLC]DRILLFEEDBACKVEL"].abs() < 0.1) & (df[k].abs() < 5.0)][k].mean()

        vals = d[k]# - zero_offset
        display_max = max(vals.abs().quantile(0.99), display_max)
        depth_max = df['[PLC]WIRESPOOLEDOUT'].max()
        #print(i,k, vals.values)
        if k=="[PLC]IMUYAW":
            xmin = -180
            xmax = 180
        elif  k=="[PLC]IMUROLL":
            xmin = -2
            xmax = 2
        elif  k=="[PLC]IMUPITCH":
            xmin = -2
            xmax = 2
        H, xedges, yedges = np.histogram2d(d["[PLC]WIRESPOOLEDOUT"], vals, bins=[int(depth_max/2), 50], range=[[0, depth_max], [xmin, xmax]])
        H_norm_rows = H / H.max(axis=1, keepdims=True)
        H_norm_rows = np.nan_to_num(H_norm_rows)
        ax.pcolormesh(yedges, xedges, H_norm_rows, rasterized=True, cmap="viridis")
        ax.set_title("%s: %s" % (string_path, k))
        ax.set_xlim(xmin, xmax)
        #ax.set_xlim(-display_max, display_max)
        #fig.title("Angle per Depth")
        ax.set_ylabel("Wire spooled out [m]")
        ax.set_xlabel("Angle [deg]")    
    yaw_rad = np.deg2rad(d["[PLC]IMUYAW"].values)
    pitch_rad= np.deg2rad(d["[PLC]IMUPITCH"].values)
    roll_rad = np.deg2rad(d["[PLC]IMUROLL"].values)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    display_max = 0
    for i, k in enumerate(["[PLC]IMUPITCH", "[PLC]IMUROLL"]):
        d = df[(df[k].abs() < 2) & (df["[PLC]WIRESPOOLEDOUT"] > 2)]
        # subtract the mean when the drill is hanging freely above the hole
        zero_offset = df[(df["[PLC]WIRESPOOLEDOUT"] < 1) & (df["[PLC]CABLESPEED"].abs() < 0.1) & (df["[PLC]DRILLFEEDBACKVEL"].abs() < 0.1) & (df[k].abs() < 5.0)][k].mean()

        vals = d[k] - zero_offset
        display_max = max(vals.abs().quantile(0.99), display_max)
        depth_max = df['[PLC]WIRESPOOLEDOUT'].max()
        H, xedges, yedges = np.histogram2d(d["[PLC]WIRESPOOLEDOUT"], d[k] - zero_offset, bins=[int(depth_max/2), 50], range=[[0, depth_max], [-2, 2]])
        H_norm_rows = H / H.max(axis=1, keepdims=True)
        H_norm_rows = np.nan_to_num(H_norm_rows)
        axes[i].pcolormesh(yedges, xedges, H_norm_rows, rasterized=True, cmap="viridis")
        axes[i].set_title("%s" % k)
    for ax in axes:
        ax.set_xlim(-display_max, display_max)
    fig.suptitle("%s: Angle per Depth" % string_path)
    fig.supylabel("Wire spooled out [m]")
    fig.supxlabel("Angle [deg]")

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    display_max = 0
    for i, k in enumerate(["hole_pitch", "hole_roll"]):
        d = df[(df[k].abs() < 2) & (df["[PLC]WIRESPOOLEDOUT"] > 2)]
        # subtract the mean when the drill is hanging freely above the hole
        zero_offset = df[(df["[PLC]WIRESPOOLEDOUT"] < 1) & (df["[PLC]CABLESPEED"].abs() < 0.1) & (df["[PLC]DRILLFEEDBACKVEL"].abs() < 0.1) & (df[k].abs() < 5.0)][k].mean()

        vals = d[k] - zero_offset
        display_max = max(vals.abs().quantile(0.99), display_max)
        depth_max = df['[PLC]WIRESPOOLEDOUT'].max()
        H, xedges, yedges = np.histogram2d(d["[PLC]WIRESPOOLEDOUT"], d[k] - zero_offset, bins=[int(depth_max/2), 50], range=[[0, depth_max], [-2, 2]])
        H_norm_rows = H / H.max(axis=1, keepdims=True)
        H_norm_rows = np.nan_to_num(H_norm_rows)
        axes[i].pcolormesh(yedges, xedges, H_norm_rows, cmap="viridis")

        # Plot a regression line
        coeff = np.polyfit(d["[PLC]WIRESPOOLEDOUT"].values, vals.values, 1)
        fit_fn = np.poly1d(coeff)
        axes[i].plot([fit_fn(0), fit_fn(depth_max)], [0, depth_max], 'r--')
        axes[i].set_title("%s" % k)
        
    for ax in axes:
        ax.set_xlim(-display_max, display_max)
        #ax.tick_params(axis='both', labelsize=16)
        #ax.set_ylabel(ax.get_ylabel(), fontsize=16)
        #ax.set_xlabel(ax.get_xlabel(), fontsize=16)
    fig.suptitle("%s: Angle per Depth, normalized using IMU Yaw" % string_path)
    fig.supylabel("Wire spooled out [m]")
    fig.supxlabel("Angle [deg]")

    ## Take only angles when the drill is moving up
    df_down = df[(df['cutting'] == 0) & ((df['[PLC]WIRESPOOLEDOUT'].diff() > 0) | (df["[PLC]CABLESPEED"].abs() < 0.1))].dropna()
    #_plot_stability(df_down, "Downgoing subset")

    ## When the drill is moving up
    df_up = df[(df['cutting'] == 0) & ((df['[PLC]WIRESPOOLEDOUT'].diff() <= 0) | (df["[PLC]CABLESPEED"].abs() < 0.1))].dropna()
    #_plot_stability(df_up, "Upgoing subset")

    ## When the drill is drilling
    #df_cutting = df[(df['cutting'] == 1) | (df["[PLC]CABLESPEED"].abs() < 0.1)].dropna()
    #_plot_stability(df_cutting, "Drill is cutting subset")

    ## When the drill is not drilling
    #df_cutting = df[(df['cutting'] == 0) | (df["[PLC]CABLESPEED"].abs() < 0.1)].dropna()
    #_plot_stability(df_cutting, "Drill is not cutting subset")
    x_mean, y_mean, x_std, y_std, z_coords, line = {},{},{},{}, {},{}
    l1 = "down, not cutting"
    l2 = "up, not cutting"
    for df_subset, l in zip([df_down, df_up],[l1,l2]):
        # Plot a 3D trajectory for the hole
        dz = 1.0
        mean_angles_x, std_dev_angles_x, angle_bin_sizes_x = calc_binned_angles(df_subset, "hole_pitch", dz=dz)
        mean_angles_y, std_dev_angles_y, angle_bin_sizes_y = calc_binned_angles(df_subset, "hole_roll", dz=dz)
    
        z_coords[l], x_mean[l], x_std[l], y_mean[l], y_std[l] = calculate_trajectory_3d(
            mean_angles_x, std_dev_angles_x,
            mean_angles_y, std_dev_angles_y,
            dz=dz
        )
        #print("shape x mean for %s:" %l)
        #print(np.shape(x_mean[l]))
        fig, ax=plot_trajectory_3d(z_coords[l], x_mean[l], x_std[l], y_mean[l], y_std[l])
        print(ax)
        fig.suptitle('%s: %s' % (string_path, fig.get_suptitle()))
        fig.set_size_inches(14, 7)
        for ia,a in enumerate(ax):
            #a.tick_params(axis='both', labelsize=16)
            #a.set_ylabel(a.get_ylabel(), fontsize=16)
            a.set_ylim(-0.5,0.2)
            #a.set_xlabel(a.get_xlabel(), fontsize=16)
            #if (ia==0):
            #    a.set_zlabel(a.get_zlabel(), fontsize=16)
            #a.set_legend(a.get_legend(), fontsize=14)
            a.set_title("%s: %s" % (l, a.get_title()))
            #a.set_title(a.get_title(), fontsize=16)
       
        
        incl_mean, incl_std = calc_inclination(mean_angles_x, std_dev_angles_x,
                                           mean_angles_y, std_dev_angles_y)
        fig, ax = plot_inclination(z_coords[l], incl_mean, incl_std / np.sqrt(angle_bin_sizes_x))
        fig.set_size_inches(5,10)
        ax.set_title("%s (%s): %s " % (string_path,l, ax.get_title()))
    
    segments ={}
    fig, ax = plt.subplots(figsize=(5, 5))
    for l in [l1,l2]:
        
        # 2. Reshape data into continuous segments (pairs of connected points)
        points = np.array([x_mean[l], y_mean[l]]).T.reshape(-1, 1, 2)
        segments[l] = np.concatenate([points[:-1], points[1:]], axis=1)
        # 4. Plot the trajectory
        
        x_upper = x_mean[l] + x_std[l]
        x_lower = x_mean[l] - x_std[l]
        y_upper = y_mean[l] + y_std[l]
        y_lower = y_mean[l] - y_std[l]

        # 3. Construct 4-sided polygon blocks for the error band
        # Each block connects (lower_i, upper_i) to (upper_{i+1}, lower_{i+1})
        polygons = []
        for i in range(len(z_coords[l]) - 1):
            # Clockwise or counter-clockwise coordinates for a single quad block
            quad = [
                [x_lower[i], y_lower[i]],
                [x_upper[i], y_upper[i]],
                [x_upper[i+1], y_upper[i+1]],
                [x_lower[i+1], y_lower[i+1]]
            ]
            polygons.append(quad)
        
        # 4. Set up the plotting canvas
        #fig, ax = plt.subplots(figsize=(8, 6))
        norm = plt.Normalize(0,100)
        cmap_name = 'viridis'
        
        # 5. Create and style the colored error band (PolyCollection)
        # We average the z-value across the segment to align colors perfectly
        z_segments = 0.5 * (z_coords[l][:-1] + z_coords[l][1:])
        
        pc = PolyCollection(polygons, cmap=cmap_name, norm=norm, alpha=0.3)
        pc.set_array(z_segments)
        pc.set_edgecolor('none')  # Hide polygon edges to ensure a smooth blend
        ax.add_collection(pc)

        # # Add the error bars (drawn underneath the line using a muted color)
        # ax.errorbar(
        #     x_mean[l], y_mean[l], 
        #     xerr=x_std[l], 
        #     yerr=y_std[l], 
        #     fmt='none',          # 'none' ensures it only plots bars, no markers or lines
        #     ecolor='gray',       # Muted color so it doesn't fight the colormap
        #     elinewidth=1,        # Thin lines for clean visuals
        #     capsize=2,           # Small caps on the ends of error bars
        #     alpha=0.6,           # Slight transparency
        #     zorder=1             # Puts error bars behind the main trajectory line
        # )
        
        # 6. Create and style the colored trajectory line (LineCollection)
        lc = LineCollection(segments[l], cmap=cmap_name, norm=norm)
        #line[l] = ax.add_collection(lc)
        lc.set_array(z_segments)
        lc.set_linewidth(2)
        if (l == l2):
            lc.set_linestyle('--')
        else:
            lc.set_linestyle('-')
        lc.set_label(l)
        line[l] = ax.add_collection(lc) 
        

    total_x_min = np.min([np.min(x_mean[l1]-x_std[l1]),np.min(x_mean[l2]-x_std[l2])])
    total_x_max = np.max([np.max(x_mean[l1]+x_std[l1]),np.max(x_mean[l2]+x_std[l2])])
    total_y_min = np.min([np.min(y_mean[l1]-y_std[l1]),np.min(y_mean[l2]-y_std[l2])])
    total_y_max = np.max([np.max(y_mean[l1]+y_std[l1]),np.max(y_mean[l2]+y_std[l2])])


    # 5. Set axis limits dynamically to fit the data
    ax.set_xlim(total_x_min - 0.01, total_x_max + 0.01)
    ax.set_ylim(total_y_min - 0.01, total_y_max + 0.01)

    # 3. Combine lines
    combined_segments = segments[l1]# + segments[l2]
    combined_lc = LineCollection(combined_segments, cmap=cmap_name, norm=norm)

    # Add colorbar and labels
    cbar = fig.colorbar(combined_lc, ax=ax)


    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.title('%s: Trajectory x vs y colored by depth' % string_path)#, fontsize=16)
    plt.legend()


    
    with matplotlib.backends.backend_pdf.PdfPages(out_path) as pdf:
        for fig in range(figures_start_idx,  plt.gcf().number + 1):
            pdf.savefig(fig)
            plt.close(fig)
    

In [8]:
def plot(folder, start_date, end_date, out_file):
    print(out_file)
    file_label = out_file.split("/")[1].replace("_inclination.pdf", "")
    out_file = Path(out_file)
    out_file = out_file.with_name(out_file.name.replace(":", "-"))
    mr = MultiLogReader.find_files(folder, start_date, end_date)
    df = mr.as_df()

    if out_file is None:
        end_date_str = ""
        if end_date is not None:
            end_date_str = f"_{end_date}"
        out_file = Path(f"BigRAID_{start_date}{end_date_str}.pdf")

    #_plot_inclination_(df,file_label, out_file)
    plot_trajectory_coordinates(df,out_file)

In [13]:
import json
import subprocess
import os
from os.path import isfile, join
year = "2025"
with open("%s-Drill-Log.json" % year) as f:
    data = json.load(f)

#folder = r"datalog/"
folder = r"/Users/delia/Library/CloudStorage/OneDrive-SharedLibraries-NERC/BAS BigRAID - Documents/Season Reports/%s/DataLog/"%year

folder = os.path.realpath(folder)
print(folder)
onlyfiles = [f for f in os.listdir(folder) if isfile(join(folder, f))]
#print(onlyfiles)

for i,d in enumerate(data):
    print(d)
    filename = f"%s-Plots/%02i_Site%02i_%02i_%s_inclination.pdf" % (year,d["number"], d["site"], d["hole"], d["geoloc"])
    plot(folder, d["start"], d["end"], filename)
    plt.close()


/Users/delia/Library/CloudStorage/OneDrive-SharedLibraries-NERC/BAS BigRAID - Documents/Season Reports/2025/DataLog
{'number': 2, 'site': 25, 'hole': 1, 'geoloc': 'South', 'start': '2025-06-04', 'end': '2025-06-06'}
2025-Plots/02_Site25_01_South_inclination.pdf
02_Site25_01_South
shape x mean for down, not cutting:
(98,)


/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:180: UserWarning: The figure layout has changed to tight
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: divide by zero encountered in divide
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: invalid value encountered in multiply
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:230: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.96])


(<Axes3D: title={'center': '3D Hole Trajectory'}, xlabel='X Position [m]', ylabel='Y Position [m]', zlabel='Z Position (Depth) [m]'>, <Axes: title={'center': 'X-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='X Position [m]'>, <Axes: title={'center': 'Y-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='Y Position [m]'>)
shape x mean for up, not cutting:
(98,)


/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:180: UserWarning: The figure layout has changed to tight
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: divide by zero encountered in divide
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: invalid value encountered in multiply
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:230: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.96])


(<Axes3D: title={'center': '3D Hole Trajectory'}, xlabel='X Position [m]', ylabel='Y Position [m]', zlabel='Z Position (Depth) [m]'>, <Axes: title={'center': 'X-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='X Position [m]'>, <Axes: title={'center': 'Y-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='Y Position [m]'>)
np.shape(points)
(98, 1, 2)
np.shape(segments[down, not cutting])
(97, 2, 2)
np.shape(points)
(98, 1, 2)
np.shape(segments[up, not cutting])
(97, 2, 2)
{'number': 3, 'site': 25, 'hole': 2, 'geoloc': 'East', 'start': '2025-06-07', 'end': '2025-06-10T14:00:00'}
2025-Plots/03_Site25_02_East_inclination.pdf
Error decoding record timestamp: time_str=b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Error decoding record timestamp: time_str=b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Error decoding record timestamp: time_str=b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Error decoding 

...

shape x mean for down, not cutting:
(99,)


/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:180: UserWarning: The figure layout has changed to tight
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: divide by zero encountered in divide
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: invalid value encountered in multiply
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:230: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.96])


(<Axes3D: title={'center': '3D Hole Trajectory'}, xlabel='X Position [m]', ylabel='Y Position [m]', zlabel='Z Position (Depth) [m]'>, <Axes: title={'center': 'X-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='X Position [m]'>, <Axes: title={'center': 'Y-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='Y Position [m]'>)
shape x mean for up, not cutting:
(100,)


/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:180: UserWarning: The figure layout has changed to tight
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: divide by zero encountered in divide
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:107: RuntimeWarning: invalid value encountered in multiply
  inclination_std = (1/np.tan(inclination_mean)) * np.sqrt(
/Users/delia/Library/CloudStorage/OneDrive-UW-Madison/RNO-G/data_analysis/git/bigraid_log_reader/hole_trajectory.py:230: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.96])


(<Axes3D: title={'center': '3D Hole Trajectory'}, xlabel='X Position [m]', ylabel='Y Position [m]', zlabel='Z Position (Depth) [m]'>, <Axes: title={'center': 'X-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='X Position [m]'>, <Axes: title={'center': 'Y-Z Plane Projection'}, xlabel='Z Position (Depth) [m]', ylabel='Y Position [m]'>)
np.shape(points)
(99, 1, 2)
np.shape(segments[down, not cutting])
(98, 2, 2)
np.shape(points)
(100, 1, 2)
np.shape(segments[up, not cutting])
(99, 2, 2)
